In [1]:
%load_ext autoreload
%autoreload 2


In [146]:
%reload_ext autoreload


In [156]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from skfda import FDataGrid
from skfda.representation.basis import BSplineBasis
from skfda.preprocessing.smoothing import BasisSmoother
from skfda.misc.regularization import L2Regularization
from skfda.misc.operators import LinearDifferentialOperator


from src.Smooth_Method.Smoothing_Spline import registration, lambda_finding, compute_smoothed_train
from src.MFPCA import run_mfpca
from src.Label import fit_gmm, label_merged, filter_candidates, take_iv, Youden_merged
from src.Smooth_Method.Regression_Spline import compute_smoothed_test
from src.Youden import ff
from src.result_check import check_rmse, sweep_k
sensor_names = [
    "T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr",
    "Ps30", "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nfdmd", "PCNfRdmd", "W31", "W32"
]
base_names = ['unit_number', 'cycles', 'op1', 'op2', 'op3']
sensor_columns = base_names + sensor_names 

base_keep_names = ['unit_number', 'cycles']
SENSORS = ["T24", "T30", "T50", "P30", "Ps30", "phi", "BPR", "W31", "W32"]
sensor_keep_columns = base_keep_names + SENSORS
train_df = pd.read_csv('Data/train_FD001.txt', sep='\\s+', header=None, names=sensor_columns)
train_df = train_df[sensor_keep_columns]

test_df = pd.read_csv('Data/test_FD001.txt', sep='\\s+', header=None, names=sensor_columns)
test_df = test_df[sensor_keep_columns]

check_df = pd.read_csv('Data/RUL_FD001.txt', sep='\\s+', header=None, names=['RUL'])
check_df['unit_number'] = np.arange(1, len(check_df) + 1)



In [151]:
test_df

,unit_number,cycles,T24,T30,T50,P30,Ps30,phi,BPR,W31,W32
0,1,1,643.02,1585.29,1398.21,553.90,47.20,521.72,8.4052,38.86,23.3735
1,1,2,641.71,1588.45,1395.42,554.85,47.50,522.16,8.3803,39.02,23.3916
2,1,3,642.46,1586.94,1401.34,554.11,47.50,521.97,8.4441,39.08,23.4166
3,1,4,642.44,1584.12,1406.42,554.07,47.28,521.38,8.3917,39.00,23.3737
4,1,5,642.51,1587.19,1401.92,554.16,47.31,522.15,8.4031,38.99,23.4130
...,...,...,...,...,...,...,...,...,...,...,...
13091,100,194,643.24,1599.45,1415.79,553.41,47.69,520.69,8.4715,38.65,23.1974
13092,100,195,643.22,1595.69,1422.05,553.22,47.60,521.05,8.4512,38.57,23.2771
13093,100,196,643.44,1593.15,1406.82,553.04,47.57,521.18,8.4569,38.62,23.2051
13094,100,197,643.26,1594.99,1419.36,553.37,47.61,521.33,8.4711,38.66,23.2699


In [33]:
train_df

,unit_number,cycles,T24,T30,T50,P30,Ps30,phi,BPR,W31,W32
0,1,1,641.82,1589.70,1400.60,554.36,47.47,521.66,8.4195,39.06,23.4190
1,1,2,642.15,1591.82,1403.14,553.75,47.49,522.28,8.4318,39.00,23.4236
2,1,3,642.35,1587.99,1404.20,554.26,47.27,522.42,8.4178,38.95,23.3442
3,1,4,642.35,1582.79,1401.87,554.45,47.13,522.86,8.3682,38.88,23.3739
4,1,5,642.37,1582.85,1406.22,554.00,47.28,522.19,8.4294,38.90,23.4044
...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,643.49,1597.98,1428.63,551.43,48.07,519.49,8.4956,38.49,22.9735
20627,100,197,643.54,1604.50,1433.58,550.86,48.04,519.68,8.5139,38.30,23.1594
20628,100,198,643.42,1602.46,1428.18,550.94,48.09,520.01,8.5646,38.44,22.9333
20629,100,199,643.23,1605.26,1426.53,550.68,48.39,519.67,8.5389,38.29,23.0640


In [34]:
train_df = registration(train_df)

In [ ]:
summary_lambdas, unit_cache = lambda_finding(train_compare_df, SENSORS, cycle_col="t_registered")

  sensor    lambda        gcv
0    T24  0.012581   0.093138
1    T30  0.014806  16.366782
2    T50  0.005430  16.587771
3    P30  0.005976   0.170488
4   Ps30  0.004022   0.010600
5    phi  0.004822   0.093728
6    BPR  0.008878   0.000413
7    W31  0.010000   0.010471
8    W32  0.010113   0.003694


In [55]:
train_smoothed = compute_smoothed_train(train_df, summary_lambdas, unit_cache, SENSORS)

Smoothed Dataframe :
       unit_number  t_registered         T24          T30          T50  \
0                1      0.000000  642.289657  1587.434701  1400.757907   
1                1      0.005236  642.291533  1587.410602  1400.722768   
2                1      0.010471  642.293403  1587.386568  1400.687608   
3                1      0.015707  642.295262  1587.362617  1400.652604   
4                1      0.020942  642.297102  1587.338764  1400.617936   
...            ...           ...         ...          ...          ...   
20626          100      0.979899  643.483922  1600.628724  1427.493349   
20627          100      0.984925  643.496300  1600.781270  1427.825041   
20628          100      0.989950  643.508681  1600.933899  1428.157030   
20629          100      0.994975  643.521066  1601.086555  1428.489083   
20630          100      1.000000  643.533456  1601.239188  1428.820966   

              P30       Ps30         phi       BPR        W31        W32  
0      554.1326

In [56]:
rho_score, exp_var = run_mfpca(train_smoothed, SENSORS)

   mfpc  explained_var_pct  cumulative_pct
0     1          95.084146       95.084146
1     2           2.107639       97.191786
2     3           0.753925       97.945710
3     4           0.481677       98.427387
4     5           0.427724       98.855111
5     6           0.344398       99.199509
6     7           0.303259       99.502769
7     8           0.193182       99.695951
8     9           0.160539       99.856490
9    10           0.072833       99.929323


In [75]:
label_train = fit_gmm(train_smoothed, rho_score)

label
0    65
1    35
Name: count, dtype: int64


In [63]:
iv_train = take_iv(train_smoothed, SENSORS)

In [76]:
youden_info = Youden_merged(label_train, iv_train)

In [78]:
iv_test = take_iv(test_df, SENSORS)

In [82]:
label_test = label_merged(test_df, iv_test, youden_info, SENSORS)

label
0    62
1    38
Name: count, dtype: int64


In [110]:
candidates = filter_candidates(train_df, test_df, label_train, label_test)

[filter_candidates] 1/100 units fell back to the opposite group due to insufficient candidates :
  - unit 49: original_group=1, n_own_group=35, n_opposite_valid=4


In [127]:
smoothed_test = compute_smoothed_test(candidates, train_df, test_df, SENSORS)

In [141]:
result = ff(smoothed_test, train_df, candidates, k=8)

In [170]:
compare_df, rmse = check_rmse(result, check_df)

In [171]:
# compare_df[compare_df['unit_number']==20]
compare_df.head(20)

,unit_number,RUL_pred_mean,RUL_pred_median,RUL
0,1,150.125,157.5,112
1,2,143.625,138.0,98
2,3,70.875,53.0,69
3,4,82.000,71.5,82
4,5,85.625,77.0,91
5,6,93.375,92.0,93
6,7,83.875,85.0,91
7,8,61.625,55.5,95
8,9,163.250,163.5,111
9,10,65.125,64.5,96


In [159]:
rmse


{'mean': np.float64(25.987097158974876),
 'median': np.float64(27.737654911689994)}

In [172]:
def round_predictions_and_recount(df):
    """
    Hàm làm tròn các cột RUL dự đoán về số nguyên 
    và tính toán lại cột matches_count.
    """
    # 1. Làm tròn và ép kiểu về số nguyên (int)
    df['RUL_pred_mean'] = df['RUL_pred_mean'].round().astype(int)
    df['RUL_pred_median'] = df['RUL_pred_median'].round().astype(int)
    
    # 2. Đếm lại số lượng trùng khớp sau khi đã làm tròn
    df['matches_count_mean'] = (df['RUL_pred_mean'] == df['RUL']).astype(int)
    df['median']=(df['RUL_pred_median'] == df['RUL']).astype(int)
    
    return df

# Cách gọi hàm để áp dụng lên dataframe của bạn:
compare_df = round_predictions_and_recount(compare_df)

# Xem kết quả 5 dòng đầu
compare_df['matches_count_mean'].value_counts()


matches_count_mean
0    98
1     2
Name: count, dtype: int64

In [173]:
compare_df['median'].value_counts()

median
0    99
1     1
Name: count, dtype: int64

In [175]:
compare_df

,unit_number,RUL_pred_mean,RUL_pred_median,RUL,matches_count_mean,median
0,1,150,158,112,0,0
1,2,144,138,98,0,0
2,3,71,53,69,0,0
3,4,82,72,82,1,0
4,5,86,77,91,0,0
...,...,...,...,...,...,...
95,96,146,140,137,0,0
96,97,74,70,82,0,0
97,98,79,76,59,0,0
98,99,120,114,117,0,0
